# Module 2 Extensions — Preprocessing, Learning Curves & Uncertainty

This notebook covers three additions that go beyond the original assignment
and make the pipeline closer to production-ready:

1. **Spectral preprocessing** — baseline correction and cosmic ray removal.
   What you'd have to do with genuinely raw spectra before any analysis.

2. **Learning curves** — how much data do we actually need? Where does
   performance plateau, and which classifiers overfit?

3. **Uncertainty quantification** — instead of hard cancer/healthy labels,
   output a confidence score and flag uncertain predictions rather than
   forcing a binary decision when the model isn't sure.

All run on the real Yin et al. (2021) COVID-19 serum Raman dataset.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from module2_raman.loader import load_covid_raman, load_synthetic
from module2_raman.preprocessing import (
    preprocess_spectra, detect_cosmic_rays, remove_cosmic_rays,
    asymmetric_least_squares, plot_preprocessing
)
from module2_raman.pca_analysis import optimise_spectral_range
from module2_raman.classification import (
    extract_features, build_classifiers,
    plot_learning_curves,
    predict_with_confidence, plot_confidence_distribution,
)

plt.rcParams['figure.dpi'] = 110
print('Ready.')

## Part 1 — Spectral Preprocessing

### Why the COVID-19 dataset doesn't need this — but real data does

The Yin et al. data was already baseline-corrected by the authors before
publication. We demonstrate preprocessing on synthetic raw spectra so
you can see what the pipeline does, then verify it doesn't degrade the
real data quality when applied.

### The two problems in raw Raman spectra

**Fluorescence background:** Biological tissue absorbs photons and
re-emits them at longer wavelengths (fluorescence). This produces a
broad, slowly-varying background that can be 10–100× larger than the
Raman signal. It must be removed to see the Raman peaks clearly.

**Cosmic rays:** High-energy charged particles (from cosmic rays or
radioactive decay) occasionally hit the CCD detector. Each impact
deposits so much energy that it saturates 1–3 pixels, creating a spike
that looks like an impossibly sharp, intense Raman peak. If not removed,
it will be selected as a key feature and corrupt the analysis.

In [ ]:
# ── Simulate a raw spectrum with fluorescence + cosmic ray ─────────────────
# Real raw Raman spectra look like this before any processing.
# We use the synthetic loader's spectral shape as the Raman signal,
# then add a realistic fluorescence baseline on top.

ds_synth = load_synthetic(n_cancer=10, n_healthy=10)
raman_shifts = ds_synth.raman_shifts

# Take one spectrum and un-normalise it to raw scale
raman_signal = ds_synth.cancer[0] * 500   # scale to raw counts

# Add a realistic fluorescence baseline (broad exponential decay)
n = len(raman_shifts)
fluorescence = 3000 * np.exp(-np.linspace(0, 3, n)) + 500

# Add a cosmic ray spike at 1100 cm⁻¹
cosmic_idx = np.argmin(np.abs(raman_shifts - 1100))
raw_spectrum = raman_signal + fluorescence
raw_with_spike = raw_spectrum.copy()
raw_with_spike[cosmic_idx] += 8000   # cosmic ray spike

print(f'Raman signal range:    {raman_signal.min():.0f}–{raman_signal.max():.0f} counts')
print(f'Fluorescence range:    {fluorescence.min():.0f}–{fluorescence.max():.0f} counts')
print(f'Cosmic ray intensity:  {raw_with_spike[cosmic_idx]:.0f} counts at {raman_shifts[cosmic_idx]:.0f} cm⁻¹')

In [ ]:
# ── Step 1: Detect and remove the cosmic ray ───────────────────────────────
spike_mask = detect_cosmic_rays(raw_with_spike, threshold=5.0)
print(f'Cosmic rays detected at: {raman_shifts[spike_mask].tolist()} cm⁻¹')

spectrum_no_spike = remove_cosmic_rays(raw_with_spike, threshold=5.0)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(raman_shifts, raw_with_spike,    'r-',  lw=1,   alpha=0.7, label='Raw (with cosmic ray)')
ax.plot(raman_shifts, spectrum_no_spike, 'b-',  lw=1.5, label='After cosmic ray removal')
ax.axvline(raman_shifts[cosmic_idx], color='red', lw=1, linestyle=':')
ax.set_xlabel('Raman shift (cm⁻¹)')
ax.set_ylabel('Intensity (counts)')
ax.set_title('Cosmic ray removal')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Step 2: Estimate and remove the fluorescence baseline (ALS) ────────────
baseline = asymmetric_least_squares(spectrum_no_spike, lam=1e5, p=0.01)
corrected = np.clip(spectrum_no_spike - baseline, 0, None)

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

axes[0].plot(raman_shifts, spectrum_no_spike, 'steelblue', lw=1, label='Spike-free spectrum')
axes[0].plot(raman_shifts, baseline, 'r--', lw=2, label='ALS baseline estimate')
axes[0].plot(raman_shifts, fluorescence, 'g:', lw=1.5, label='True fluorescence (known)')
axes[0].set_ylabel('Intensity (counts)')
axes[0].set_title('ALS baseline estimation')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

axes[1].plot(raman_shifts, corrected,     'steelblue', lw=1.5, label='ALS corrected')
axes[1].plot(raman_shifts, raman_signal,  'g--',       lw=1,   label='True Raman signal (known)', alpha=0.7)
axes[1].set_ylabel('Intensity (counts)')
axes[1].set_title('After baseline removal vs true Raman signal')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

residual = np.abs(corrected - raman_signal)
axes[2].fill_between(raman_shifts, residual, alpha=0.4, color='orange', label='Residual error')
axes[2].set_xlabel('Raman shift (cm⁻¹)')
axes[2].set_ylabel('Error (counts)')
axes[2].set_title('Baseline correction residual')
axes[2].legend(fontsize=9)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Mean residual error: {residual.mean():.2f} counts  '
      f'({residual.mean()/raman_signal.max()*100:.1f}% of peak intensity)')

In [ ]:
# ── Apply preprocessing to real data and check it doesn't degrade quality ──
# The COVID data is already clean, so preprocessing should change it
# minimally. If MAE after preprocessing is very small, it's safe to apply.

ds = load_covid_raman()
processed_cancer = preprocess_spectra(
    ds.cancer,
    remove_spikes=True,
    correct_fluorescence=True,  # ALS will find a near-flat baseline on already-clean data
    normalise=True,
)
change = np.abs(processed_cancer - ds.cancer).mean()
print(f'Mean change per spectrum after preprocessing: {change:.5f}')
print('(Small value confirms the data was already clean — preprocessing is safe to apply)')

## Part 2 — Learning Curves

### What learning curves tell you that accuracy alone can't

The accuracy metrics in notebook 03 told us how well each classifier
performs on the full dataset. Learning curves answer a different question:
**how does performance change as the training set grows?**

This matters because:
- If the CV curve is still rising at max data, collecting more samples
  would improve performance
- A large gap between training and CV curves means overfitting
- A curve that plateaus early means the classifier has a capacity ceiling
  — a more expressive model might help more than more data

In [ ]:
from module2_raman.pca_analysis import optimise_spectral_range

pca_results = optimise_spectral_range(ds.cancer, ds.healthy, ds.raman_shifts)
wn_min, wn_max = pca_results[0].spectral_range
X, y = extract_features(ds.cancer, ds.healthy, ds.raman_shifts,
                         wn_min=wn_min, wn_max=wn_max)
print(f'Feature matrix: {X.shape}  ({X.shape[0]} samples, {X.shape[1]} PCA features)')
print(f'Spectral range: {wn_min:.0f}–{wn_max:.0f} cm⁻¹')

In [ ]:
# This takes ~30–60 seconds — each classifier is trained 8 times
# at different data sizes, repeated across 4 CV folds
print('Computing learning curves (may take ~30s)...')
fig = plot_learning_curves(X, y, n_folds=4)
plt.show()

# What to discuss:
# - Which classifiers show a large train/CV gap? (overfitting)
# - Which CV curves are still rising? (would benefit from more data)
# - Which plateau quickly? (capacity-limited, not data-limited)

## Part 3 — Uncertainty Quantification

### Why hard labels are not enough in clinical settings

Every classifier we've built returns one of two outputs: cancer or healthy.
But classifiers that support `predict_proba()` actually compute a probability
for each class — the hard label is just whichever probability is higher.

A prediction of P(disease)=0.51 and P(disease)=0.99 both produce the same
label, but they should absolutely not be treated the same way by a surgeon.

**The approach here:**
- If confidence (probability of the predicted class) ≥ threshold → report the label
- If confidence < threshold → report 'uncertain', refer for additional testing

This is called **selective prediction** or **prediction with abstention**.
It trades coverage (fewer predictions made) for reliability (higher accuracy
on the predictions that are made).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Use SVM — it tends to give well-spread probability estimates
clf = build_classifiers()['SVM (RBF)']

thresholds = [0.60, 0.75, 0.90]

print(f'{'Threshold':>12}  {'Coverage':>10}  {'Acc (certain)':>15}  {'Uncertain cases':>17}')
print('-' * 60)
for t in thresholds:
    pred = predict_with_confidence(clf, X_train, y_train, X_test, threshold=t)
    coverage     = pred.certain.mean() * 100
    acc_certain  = (pred.label[pred.certain] == y_test[pred.certain]).mean() if pred.certain.any() else 0
    n_uncertain  = (~pred.certain).sum()
    print(f'{t:>12.0%}  {coverage:>9.1f}%  {acc_certain:>15.3f}  {n_uncertain:>17} samples')

In [ ]:
# ── Visualise confidence for the 75% threshold ─────────────────────────────
pred = predict_with_confidence(clf, X_train, y_train, X_test, threshold=0.75)

fig = plot_confidence_distribution(
    pred, y_test,
    threshold=0.75,
    title='SVM (RBF) — Confidence Distribution on Test Set',
)
plt.show()

# The left panel shows that most errors cluster near the decision boundary
# (low confidence) — the model knows when it's unsure.
# The right panel shows the probability space: well-separated clusters
# indicate good calibration.

In [ ]:
# ── Compare all classifiers at 75% threshold ───────────────────────────────
print(f'{'Model':>22}  {'Coverage':>10}  {'Acc (certain)':>14}  {'Acc (all)':>10}')
print('-' * 64)

for name, clf in build_classifiers().items():
    try:
        pred = predict_with_confidence(clf, X_train, y_train, X_test, threshold=0.75)
        cov  = pred.certain.mean() * 100
        acc_c = (pred.label[pred.certain] == y_test[pred.certain]).mean() if pred.certain.any() else float('nan')
        acc_a = (pred.label == y_test).mean()
        print(f'{name:>22}  {cov:>9.1f}%  {acc_c:>14.3f}  {acc_a:>10.3f}')
    except AttributeError:
        print(f'{name:>22}  (no probability output)')

## Summary

| Extension | What it adds |
|-----------|-------------|
| **Preprocessing** | Handles raw spectra with fluorescence baseline and cosmic ray contamination — required for real instrument data |
| **Learning curves** | Reveals overfitting, data requirements, and capacity limits — goes beyond headline accuracy |
| **Uncertainty quantification** | Makes predictions clinically usable — flags uncertain cases rather than forcing a binary output when the model is unsure |

All three are standard in applied ML/spectroscopy work but go beyond
what the original MATLAB assignment covered.